# Phi-4 SFT Evaluation Benchmark (Kaggle, T4)

Evaluates the **fine-tuned** Phi-4 (SFT adapter) on all 6 SE tasks.
Compares against the base Phi-4 baseline results already in `results/baseline/phi4mini/`.

Loads base model in FP16 + PEFT adapter from an attached Kaggle dataset.
Recommended adapter: **checkpoint-203** (epoch 1, best val loss 0.5710).

In [ ]:
# Environment verification
import sys, subprocess
def pip(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", pkg])
try:
    import peft
    print("peft", peft.__version__)
except Exception as e:
    print("installing peft:", e)
    pip("peft")
import torch, transformers, accelerate
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0), "vram GB", round(torch.cuda.get_device_properties(0).total_memory/1e9,2))


In [ ]:
# Kaggle-only storage paths. No Google Drive, no Colab, no cloud sync.
import glob
import os
import socket
import uuid
from datetime import datetime, timezone

IN_KAGGLE = os.path.isdir("/kaggle/working") or bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
if not IN_KAGGLE:
    raise RuntimeError(
        "This notebook is Kaggle-only. Open it in a Kaggle Notebook with a GPU "
        "accelerator; do not run it on Colab or on a local machine."
    )

PLATFORM = "kaggle"
INPUT_ROOT = "/kaggle/input"
WORKDIR = "/kaggle/working"
RESULTS_ROOT = os.path.join(WORKDIR, "results")
CHECKPOINTS_ROOT = RESULTS_ROOT
ZIP_DIR = WORKDIR

# Keep Hugging Face downloads under the disposable working tree as well.
# The export ZIP only contains results/, so the cache is never exported.
HF_CACHE_ROOT = os.path.join(WORKDIR, ".hf_cache")
os.environ.setdefault("HF_HOME", HF_CACHE_ROOT)
os.environ.setdefault("HF_HUB_CACHE", os.path.join(HF_CACHE_ROOT, "hub"))

TASKS = [
    "code_generation",
    "code_explanation",
    "bug_detection",
    "code_refactoring",
    "unit_test_generation",
    "code_completion",
]

def _valid_benchmark_root(root):
    # The dataset root must contain the model-facing test files, the private
    # references and the prompt registry. Nothing is written to this tree.
    test_dir = os.path.join(root, "benchmark", "datasets", "test")
    private_dir = os.path.join(root, "benchmark", "datasets", "private", "test")
    prompts = os.path.join(root, "benchmark", "prompts", "prompt_registry.py")
    if not (os.path.isdir(test_dir) and os.path.isdir(private_dir) and os.path.isfile(prompts)):
        return False
    return all(os.path.isfile(os.path.join(test_dir, f"{task}.jsonl")) for task in TASKS)

# A Kaggle dataset is mounted at /kaggle/input/<dataset-slug>/. The benchmark
# folder may be the dataset root or one or two levels below it. Only directory
# names and file presence are checked here.
_candidates = []
for _pattern in (
    os.path.join(INPUT_ROOT, "*", "benchmark"),
    os.path.join(INPUT_ROOT, "*", "*", "benchmark"),
    os.path.join(INPUT_ROOT, "*", "*", "*", "benchmark"),
):
    for _benchmark_dir in sorted(glob.glob(_pattern)):
        _root = os.path.dirname(_benchmark_dir)
        if _root not in _candidates and _valid_benchmark_root(_root):
            _candidates.append(_root)

# Explicit writable fallback for a manual upload, clearly flagged. Kaggle
# input datasets are read-only and are always preferred.
_working_fallback = os.path.join(WORKDIR, "slm-benchmark")
if not _candidates and _valid_benchmark_root(_working_fallback):
    _candidates.append(_working_fallback)

if not _candidates:
    raise FileNotFoundError(
        "Benchmark dataset not found under /kaggle/input. Attach a private "
        "Kaggle Dataset whose root contains benchmark/datasets/test/*.jsonl, "
        "benchmark/datasets/private/test/*.jsonl and "
        "benchmark/prompts/prompt_registry.py."
    )

BENCHMARK_ROOT = _candidates[0]
INPUT_IS_READ_ONLY = BENCHMARK_ROOT.startswith(INPUT_ROOT + os.sep)
if not INPUT_IS_READ_ONLY:
    print("WARNING: benchmark files are under /kaggle/working (writable).")
    print("         Prefer attaching them as a read-only /kaggle/input dataset.")

DATASETS_TEST = os.path.join(BENCHMARK_ROOT, "benchmark", "datasets", "test")
DATASETS_PRIVATE_TEST = os.path.join(BENCHMARK_ROOT, "benchmark", "datasets", "private", "test")
PROMPTS_DIR = os.path.join(BENCHMARK_ROOT, "benchmark", "prompts")

MODEL_DIR_NAMES = ["qwen15", "qwen3", "phi4mini"]
for _mode_dir in ("smoke_test", "baseline"):
    for _model_dir in MODEL_DIR_NAMES:
        os.makedirs(os.path.join(RESULTS_ROOT, _mode_dir, _model_dir), exist_ok=True)
os.makedirs(os.path.join(RESULTS_ROOT, "metadata"), exist_ok=True)

assert RESULTS_ROOT.startswith(WORKDIR + os.sep), RESULTS_ROOT
assert not RESULTS_ROOT.startswith(INPUT_ROOT + os.sep), RESULTS_ROOT

print("PLATFORM =", PLATFORM)
print("Benchmark root candidates =", _candidates)
print("BENCHMARK_ROOT =", BENCHMARK_ROOT)
print("INPUT_IS_READ_ONLY =", INPUT_IS_READ_ONLY)
print("RESULTS_ROOT =", RESULTS_ROOT)
print("CHECKPOINTS_ROOT =", CHECKPOINTS_ROOT)
print("ZIP_DIR =", ZIP_DIR)
print("HF_CACHE_ROOT =", HF_CACHE_ROOT)
print("LOCAL_DOWNLOAD_HINT =", os.path.join(os.environ.get("USERPROFILE", "%USERPROFILE%"), "Downloads", "slm-benchmark"))


In [ ]:
# Benchmark configuration (Kaggle-only) - SFT variant
import os, random, glob, socket, uuid
from datetime import datetime, timezone

MODEL_CHOICES = [
    "microsoft/Phi-4-mini-instruct (base + SFT adapter)",
]

# Base model ID (for metadata consistency with baseline)
MODEL_ID = "microsoft/Phi-4-mini-instruct"

# SFT-specific identifiers (distinct from base run)
SHORT = "phi-4-mini-sft"
MODEL_DIR = "phi4mini_sft"

MODE = "full"
if MODE == "baseline":
    print("MODE='baseline' is a legacy alias; using MODE='full'.")
    MODE = "full"
SMOKE_PER_TASK = 3
assert MODE in ("smoke_test", "full"), MODE
MODE_DIR = {"full": "baseline", "smoke_test": "smoke_test"}
RESULTS_MODE_DIR = MODE_DIR[MODE]

GEN_CONFIG = {
    "temperature": 0.0,
    "do_sample": False,
    "top_p": 1.0,
    "max_new_tokens": 512,
    "repetition_penalty": 1.0,
}
SEED = 42
random.seed(SEED)

# Base model metadata (same as baseline for loading)
BASE_META = {
    "trust_remote_code": True,
    "attn": "eager",
}

# Optional read-only Kaggle Models inputs for the BASE model.
BASE_MODEL_SOURCES = {
    "microsoft/Phi-4-mini-instruct": [
        "/kaggle/input/phi-4-mini-instruct/transformers/official-from-huggingface/1",
        "/kaggle/input/phi-4-mini-instruct/transformers/official-from-huggingface/1/phi-4-mini-instruct",
    ],
}

def resolve_base_source(model_id):
    for candidate in BASE_MODEL_SOURCES.get(model_id, []):
        if os.path.isfile(os.path.join(candidate, "config.json")):
            return candidate
    return model_id

BASE_MODEL_SOURCE = resolve_base_source(MODEL_ID)

# Auto-discover the SFT adapter in mounted Kaggle datasets.
ADAPTER_SOURCE = None
for p in glob.glob("/kaggle/input/**/adapter_config.json", recursive=True):
    ADAPTER_SOURCE = os.path.dirname(p)
    break
print("ADAPTER_SOURCE (auto-discovered):", ADAPTER_SOURCE)
assert ADAPTER_SOURCE is not None, "No adapter_config.json found in /kaggle/input/. Attach the SFT adapter dataset."

USE_ADAPTER = True

# Single GPU only.
SINGLE_GPU = True
DEVICE_MAP = {"": 0} if SINGLE_GPU else "auto"

MAX_INPUT_TOKENS = 8192

# Safety gate
EXECUTION_BACKEND = "none"
ALLOW_UNSANDBOXED_CODE_EXECUTION = False
TASKS_REQUIRING_EXECUTION = {
    "code_generation",
    "code_refactoring",
    "unit_test_generation",
}
assert EXECUTION_BACKEND == "none"
assert ALLOW_UNSANDBOXED_CODE_EXECUTION is False

print("Base model:", MODEL_ID, "| short:", SHORT, "| model_dir:", MODEL_DIR)
print("Base model source:", BASE_MODEL_SOURCE)
print("Adapter source:", ADAPTER_SOURCE)
print("MODE:", MODE, "| results subdir:", RESULTS_MODE_DIR)
print("Execution backend:", EXECUTION_BACKEND)

# Run ID
try:
    RUN_IDS
except NameError:
    RUN_IDS = {}
_run_key = (MODE, MODEL_DIR)
if _run_key not in RUN_IDS:
    _stamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
    RUN_IDS[_run_key] = f"{MODEL_DIR}_{MODE}_{_stamp}_{socket.gethostname()}_{uuid.uuid4().hex[:8]}"
    print("Generated RUN_ID =", RUN_IDS[_run_key])
else:
    print("Reusing RUN_ID =", RUN_IDS[_run_key])
RUN_ID = RUN_IDS[_run_key]
RUN_DIR = os.path.join(RESULTS_ROOT, RESULTS_MODE_DIR, MODEL_DIR, RUN_ID)
os.makedirs(RUN_DIR, exist_ok=True)
METADATA_DIR = os.path.join(RESULTS_ROOT, "metadata")
os.makedirs(METADATA_DIR, exist_ok=True)
assert RUN_DIR.startswith(RESULTS_ROOT + os.sep), RUN_DIR
print("RUN_ID =", RUN_ID)
print("RUN_DIR =", RUN_DIR)


In [ ]:
# Load prompt registry
import importlib.util
spec = importlib.util.spec_from_file_location("prompt_registry", os.path.join(PROMPTS_DIR, "prompt_registry.py"))
prompt_registry = importlib.util.module_from_spec(spec)
spec.loader.exec_module(prompt_registry)
build_messages = prompt_registry.build_messages
apply_chat_template = prompt_registry.apply_chat_template
print("Prompt registry loaded from", PROMPTS_DIR)


In [ ]:
# Load test datasets (model-facing only)
import json

def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

test_data = {}
for task in TASKS:
    path = os.path.join(DATASETS_TEST, f"{task}.jsonl")
    rows = load_jsonl(path)
    test_data[task] = rows
    print(task, len(rows))

print("Total test examples:", sum(len(v) for v in test_data.values()))


In [ ]:
# Load base model in FP16 + PEFT adapter on exactly one GPU.
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

if "model" in globals() or "tokenizer" in globals():
    raise RuntimeError("A model/tokenizer is already loaded. Run the unload cell first.")

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

assert torch.cuda.is_available(), "No GPU available. Set Accelerator -> GPU T4 x1."
_gpu_count = torch.cuda.device_count()
print("CUDA device count:", _gpu_count)
for _i in range(_gpu_count):
    _p = torch.cuda.get_device_properties(_i)
    print(f"  GPU {_i}: {_p.name} | {_p.total_memory / 1e9:.2f} GB")

if SINGLE_GPU:
    torch.cuda.set_device(0)
    if _gpu_count > 1:
        print("WARNING: multiple GPUs visible; device_map forces GPU 0 only.")

_gpu_name = torch.cuda.get_device_name(0)
_total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print("Active GPU:", _gpu_name, "| VRAM (GB):", round(_total_vram_gb, 2))
assert _total_vram_gb >= 15.0, f"Expected >=15 GB GPU; found {_total_vram_gb:.2f} GB."

meta = BASE_META
load_kwargs = {
    "torch_dtype": torch.float16,
    "device_map": DEVICE_MAP,
    "trust_remote_code": meta["trust_remote_code"],
    "low_cpu_mem_usage": True,
}
if meta["attn"] is not None:
    load_kwargs["attn_implementation"] = meta["attn"]

# Load tokenizer from adapter dir (preserves chat template)
print("Loading tokenizer from adapter:", ADAPTER_SOURCE)
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_SOURCE, trust_remote_code=meta["trust_remote_code"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# Load base model
print("Loading base model from", BASE_MODEL_SOURCE)
t0 = time.time()
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_SOURCE, **load_kwargs)
print(f"Base model loaded in {time.time()-t0:.1f}s")

# Apply PEFT adapter
print("Applying PEFT adapter from", ADAPTER_SOURCE)
t0 = time.time()
model = PeftModel.from_pretrained(base_model, ADAPTER_SOURCE, torch_dtype=torch.float16)
model.eval()
model.config.use_cache = True
print(f"Adapter applied in {time.time()-t0:.1f}s")

print("Model dtype:", next(model.parameters()).dtype)
print("Model device:", next(model.parameters()).device)
if torch.cuda.is_available():
    print("Allocated VRAM after load (GB):", round(torch.cuda.memory_allocated() / 1e9, 3))
    torch.cuda.reset_peak_memory_stats()


In [ ]:
# Generation helpers
import time, gc, threading

def count_tokens(text):
    return len(tokenizer.encode(text, add_special_tokens=False))

def generate_one(record):
    prompt = apply_chat_template(tokenizer, record)
    full_prompt_tokens = len(tokenizer.encode(prompt, add_special_tokens=False))
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    ).to(model.device)
    input_len = inputs["input_ids"].shape[1]
    input_truncated = full_prompt_tokens > MAX_INPUT_TOKENS
    if input_truncated:
        print(f"  WARNING: prompt truncated from {full_prompt_tokens} to {input_len} tokens")

    from transformers import TextIteratorStreamer
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    gen_kwargs = dict(
        **inputs,
        max_new_tokens=GEN_CONFIG["max_new_tokens"],
        temperature=GEN_CONFIG["temperature"],
        do_sample=GEN_CONFIG["do_sample"],
        top_p=GEN_CONFIG["top_p"],
        repetition_penalty=GEN_CONFIG["repetition_penalty"],
        pad_token_id=tokenizer.pad_token_id,
        streamer=streamer,
    )

    gen_error = []

    def _run_generate():
        try:
            with torch.inference_mode():
                model.generate(**gen_kwargs)
        except Exception as exc:
            gen_error.append(exc)
            try:
                streamer.end()
            except Exception:
                pass

    thread = threading.Thread(target=_run_generate)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t_start = time.perf_counter()
    thread.start()
    collected = []
    first_token_time = None
    for chunk in streamer:
        if first_token_time is None and chunk.strip():
            first_token_time = time.perf_counter()
        collected.append(chunk)
    thread.join()
    if gen_error:
        raise gen_error[0]
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t_end = time.perf_counter()

    output_text = "".join(collected)
    output_len = count_tokens(output_text)
    latency = t_end - t_start
    ttft_val = (first_token_time - t_start) if first_token_time is not None else None
    tps = output_len / latency if latency > 0 else 0.0
    return {
        "output": output_text,
        "input_tokens": int(input_len),
        "output_tokens": int(output_len),
        "latency_s": round(latency, 4),
        "ttft_s": round(ttft_val, 4) if ttft_val is not None else None,
        "tokens_per_s": round(tps, 2),
        "input_truncated": bool(input_truncated),
    }


In [ ]:
# Evaluation helpers - no generated code is executed.
import ast
import os
import re
import difflib
import json as _json

assert EXECUTION_BACKEND == "none", (
    "This notebook only supports EXECUTION_BACKEND='none'. "
    "Do not enable unsandboxed execution."
)

try:
    import rouge_score  # noqa: F401
except ImportError as exc:
    raise RuntimeError(
        "rouge_score is required for code_explanation; install it with "
        "!pip install -q rouge_score before running."
    ) from exc

def extract_code_block(text):
    m = re.search(r"```(?:python|py|python3)?\s*\n?(.*?)```", text, re.DOTALL | re.IGNORECASE)
    if m:
        return m.group(1).strip("\n")
    return text.strip()

def _static_syntax_ok(code):
    # Parse only; never execute the code.
    try:
        ast.parse(code)
        return True
    except (SyntaxError, ValueError):
        return False

def run_python(program, timeout=10):
    # Hard block: no isolated execution backend is available.
    raise RuntimeError(
        "Refusing to execute generated code: no isolated sandbox backend is "
        "configured. This notebook performs static checks only. Do not replace "
        "this guard with an unsandboxed subprocess."
    )

def _execution_skipped():
    return {
        "skipped": True,
        "reason": "no isolated code-execution backend; generated code was not run",
        "execution_backend": EXECUTION_BACKEND,
    }

def load_private(record_id, task):
    for line in open(os.path.join(DATASETS_PRIVATE_TEST, f"{task}.jsonl"), encoding="utf-8"):
        r = _json.loads(line)
        if r["id"] == record_id:
            return r
    raise KeyError(record_id)

def eval_code_generation(record, output):
    code = extract_code_block(output)
    result = _execution_skipped()
    result["compile_ok"] = _static_syntax_ok(code)  # static syntax only
    result["static_only"] = True
    return result

def eval_code_explanation(record, output):
    priv = load_private(record["id"], "code_explanation")
    ref = priv["expected_output"]
    try:
        from rouge_score import rouge_scorer
    except ImportError as exc:
        raise RuntimeError(
            "rouge_score is required for code_explanation; install it with "
            "!pip install -q rouge_score before running."
        ) from exc
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    rouge_l = scorer.score(ref, output)["rougeL"].fmeasure
    headings = ["## Purpose", "## Parameters", "## Return value", "## Complexity"]
    heading_ok = all(h in output for h in headings)
    return {"rouge_l": rouge_l, "headings_ok": heading_ok}

def eval_bug_detection(record, output):
    try:
        obj = _json.loads(output.strip())
        pred = bool(obj.get("has_bug"))
        parse_ok = True
    except Exception:
        m = re.search(r"\{.*\}", output, re.DOTALL)
        try:
            obj = _json.loads(m.group(0)) if m else {}
            pred = bool(obj.get("has_bug"))
            parse_ok = True
        except Exception:
            pred = None
            parse_ok = False
    priv = load_private(record["id"], "bug_detection")
    truth = bool(priv["expected_output"])
    tp = fp = tn = fn = 0
    if pred is not None:
        if pred and truth: tp = 1
        elif pred and not truth: fp = 1
        elif (not pred) and (not truth): tn = 1
        else: fn = 1
    return {"parse_ok": parse_ok, "pred": pred, "truth": truth, "tp": tp, "fp": fp, "tn": tn, "fn": fn}

def eval_code_refactoring(record, output):
    code = extract_code_block(output)
    result = _execution_skipped()
    result["compile_ok"] = _static_syntax_ok(code)  # static syntax only
    result["static_only"] = True
    result["changed"] = (re.sub(r"\s+", "", code) != re.sub(r"\s+", "", record["input"]))
    return result

def eval_unit_test_generation(record, output):
    tests = extract_code_block(output)
    result = _execution_skipped()
    result["syntax_ok"] = _static_syntax_ok(tests)
    result["n_tests"] = len(re.findall(r"def\s+test_\w+", tests))
    result["static_only"] = True
    return result

def eval_code_completion(record, output):
    # Strip any markdown code fence before taking the first completed line,
    # otherwise the fence opener becomes the prediction.
    code = extract_code_block(output)
    lines = [ln for ln in code.splitlines() if ln.strip()]
    pred = lines[0].strip() if lines else ""
    priv = load_private(record["id"], "code_completion")
    target = priv["expected_output"]
    norm_pred = re.sub(r"\s+", "", pred)
    norm_target = re.sub(r"\s+", "", target)
    exact = norm_pred == norm_target
    sim = difflib.SequenceMatcher(None, pred, target).ratio()
    return {"pred": pred, "target": target, "exact_match": exact, "edit_similarity": round(sim, 4)}


In [ ]:
# Checkpointing, run metadata and SHA256 helpers
# Every row is appended to RUN_DIR/<task>_results.jsonl immediately so an
# interrupted run resumes instead of restarting. RUN_DIR is unique per
# session, so rows from other sessions can never be overwritten.
import hashlib, json, os as _os

def sha256_file(path, chunk_size=1 << 20):
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

def result_path(task):
    return _os.path.join(RUN_DIR, f"{task}_results.jsonl")

def _read_result_rows(task):
    rows = []
    path = result_path(task)
    if not _os.path.exists(path):
        return rows
    with open(path, encoding="utf-8") as fh:
        for line_no, line in enumerate(fh, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                # A truncated last line after an interruption is not counted
                # as done; the example will be retried on resume.
                print(f"  WARN {path}:{line_no}: ignoring unparsable line")
    return rows

def load_done_ids(task):
    return {r["id"] for r in _read_result_rows(task) if "id" in r}

def append_result(task, row):
    path = result_path(task)
    with open(path, "a", encoding="utf-8") as fh:
        fh.write(json.dumps(row, ensure_ascii=False) + "\n")
        fh.flush()
        try:
            _os.fsync(fh.fileno())
        except OSError:
            pass
    write_progress()

def dataset_manifest():
    manifest = {}
    for task in TASKS:
        for base, label in ((DATASETS_TEST, "test"), (DATASETS_PRIVATE_TEST, "private_test")):
            p = _os.path.join(base, f"{task}.jsonl")
            if _os.path.exists(p):
                manifest[f"{label}/{task}"] = {
                    "path": p,
                    "sha256": sha256_file(p),
                    "bytes": _os.path.getsize(p),
                }
    return manifest

def run_metadata():
    import platform as _pl, sys as _sys
    gpu = None
    try:
        import torch as _t
        if _t.cuda.is_available():
            gpu = _t.cuda.get_device_name(0)
    except Exception:
        pass
    return {
        "run_id": RUN_ID,
        "mode": MODE,
        "mode_dir": RESULTS_MODE_DIR,
        "model_id": MODEL_ID,
        "model_source": MODEL_SOURCE,
        "short": SHORT,
        "model_dir": MODEL_DIR,
        "seed": SEED,
        "gen_config": GEN_CONFIG,
        "max_input_tokens": MAX_INPUT_TOKENS,
        "platform": PLATFORM,
        "benchmark_root": BENCHMARK_ROOT,
        "benchmark_root_read_only": INPUT_IS_READ_ONLY,
        "results_root": RESULTS_ROOT,
        "checkpoints_root": CHECKPOINTS_ROOT,
        "zip_dir": ZIP_DIR,
        "run_dir": RUN_DIR,
        "single_gpu": SINGLE_GPU,
        "device_map": str(DEVICE_MAP),
        "execution_backend": EXECUTION_BACKEND,
        "allow_unsandboxed_code_execution": ALLOW_UNSANDBOXED_CODE_EXECUTION,
        "tasks_requiring_execution": sorted(TASKS_REQUIRING_EXECUTION),
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "python": _sys.version,
        "platform_str": _pl.platform(),
        "gpu": gpu,
        "tasks": TASKS,
        "smoke_per_task": SMOKE_PER_TASK if MODE == "smoke_test" else None,
        "datasets": dataset_manifest(),
        "note": "Kaggle-only temporary filesystem. No generated code is executed. Download the ZIP before disconnecting.",
    }

def write_run_metadata():
    import json as _j
    meta = run_metadata()
    out = _os.path.join(METADATA_DIR, f"run_{RUN_ID}.json")
    with open(out, "w", encoding="utf-8") as fh:
        _j.dump(meta, fh, ensure_ascii=False, indent=2)
        fh.write("\n")
    local = _os.path.join(RUN_DIR, "metadata.json")
    with open(local, "w", encoding="utf-8") as fh:
        _j.dump(meta, fh, ensure_ascii=False, indent=2)
        fh.write("\n")
    # Dataset manifest alongside the run metadata.
    dm_out = _os.path.join(METADATA_DIR, "dataset_manifest.json")
    with open(dm_out, "w", encoding="utf-8") as fh:
        _j.dump(meta["datasets"], fh, ensure_ascii=False, indent=2)
        fh.write("\n")
    return out

def write_progress():
    import json as _j
    counts = {}
    for task in TASKS:
        rows = _read_result_rows(task)
        counts[task] = {
            "rows": len(rows),
            "errors": sum(1 for r in rows if "error" in r),
            "skipped": sum(1 for r in rows if (r.get("eval") or {}).get("skipped")),
        }
    out = _os.path.join(METADATA_DIR, f"progress_{RUN_ID}.json")
    with open(out, "w", encoding="utf-8") as fh:
        _j.dump({
            "run_id": RUN_ID, "mode": MODE, "model": MODEL_ID,
            "counts": counts, "updated_utc": datetime.now(timezone.utc).isoformat(),
        }, fh, ensure_ascii=False, indent=2)
        fh.write("\n")
    return out

def aggregate_csv():
    import pandas as pd
    rows = []
    for task in TASKS:
        for r in _read_result_rows(task):
            ev = r.get("eval", {})
            rows.append({
                "model": SHORT,
                "model_dir": MODEL_DIR,
                "mode": MODE,
                "run_id": RUN_ID,
                "task": task,
                "id": r.get("id"),
                "input_tokens": r.get("input_tokens"),
                "output_tokens": r.get("output_tokens"),
                "latency_s": r.get("latency_s"),
                "ttft_s": r.get("ttft_s"),
                "tokens_per_s": r.get("tokens_per_s"),
                "vram_gb": r.get("vram_gb"),
                "input_truncated": r.get("input_truncated"),
                "eval_skipped": bool(ev.get("skipped")),
                "model_source": r.get("model_source"),
                "eval_json": json.dumps(ev, ensure_ascii=False),
            })
    if rows:
        df = pd.DataFrame(rows)
        out = _os.path.join(RUN_DIR, f"{SHORT}_all_results.csv")
        df.to_csv(out, index=False)
        print("Wrote", out, len(df), "rows")
        return out
    return None

def write_sha256sums(root, filename="SHA256SUMS.txt"):
    entries = {}
    for dirpath, _dirs, fnames in _os.walk(root):
        for fn in sorted(fnames):
            if fn == filename:
                continue
            p = _os.path.join(dirpath, fn)
            rel = _os.path.relpath(p, root).replace(_os.sep, "/")
            entries[rel] = sha256_file(p)
    out = _os.path.join(root, filename)
    with open(out, "w", encoding="utf-8", newline="\n") as fh:
        for rel in sorted(entries):
            fh.write(f"{entries[rel]}  {rel}\n")
    return out, entries

write_run_metadata()
write_progress()
print("RUN_DIR =", RUN_DIR)
print("Metadata in", METADATA_DIR)


In [ ]:
# Main benchmark loop - runs ALL tasks for the selected model
# MODE == "full" runs every test example.
# MODE == "smoke_test" runs only the first SMOKE_PER_TASK rows per task
# (same order as test_data), so smoke results never mix with official ones.
import time

EVAL_FUNCS = {
    "code_generation": eval_code_generation,
    "code_explanation": eval_code_explanation,
    "bug_detection": eval_bug_detection,
    "code_refactoring": eval_code_refactoring,
    "unit_test_generation": eval_unit_test_generation,
    "code_completion": eval_code_completion,
}

print("Execution backend:", EXECUTION_BACKEND)
print("Generated code executed:", ALLOW_UNSANDBOXED_CODE_EXECUTION)
print("Tasks with skipped scoring:", sorted(TASKS_REQUIRING_EXECUTION))

write_run_metadata()
for task in TASKS:
    rows = test_data[task]
    if MODE == "smoke_test":
        rows = rows[:SMOKE_PER_TASK]
    done = load_done_ids(task)
    print(f"\n=== {task}: {len(rows)} total, {len(done)} done, {len(rows)-len(done)} remaining ===")
    if task in TASKS_REQUIRING_EXECUTION:
        print("  NOTE: evaluation will be marked skipped (no isolated backend).")
    for i, record in enumerate(rows, 1):
        if record["id"] in done:
            continue
        try:
            gen = generate_one(record)
            ev = EVAL_FUNCS[task](record, gen["output"])
            vram = None
            if torch.cuda.is_available():
                vram = round(torch.cuda.max_memory_allocated() / 1e9, 3)
            result_row = {
                "id": record["id"],
                "task": task,
                "model": MODEL_ID,
                "model_source": MODEL_SOURCE,
                "model_dir": MODEL_DIR,
                "mode": MODE,
                "run_id": RUN_ID,
                "input_tokens": gen["input_tokens"],
                "output_tokens": gen["output_tokens"],
                "latency_s": gen["latency_s"],
                "ttft_s": gen["ttft_s"],
                "tokens_per_s": gen["tokens_per_s"],
                "input_truncated": gen["input_truncated"],
                "vram_gb": vram,
                "output": gen["output"],
                "eval_skipped": bool(ev.get("skipped")),
                "eval": ev,
            }
            append_result(task, result_row)
            print(f"  [{i}/{len(rows)}] {record['id']} ok | lat={gen['latency_s']}s tps={gen['tokens_per_s']}")
        except Exception as e:
            err_row = {
                "id": record["id"], "task": task, "model": MODEL_ID,
                "model_source": MODEL_SOURCE,
                "model_dir": MODEL_DIR, "mode": MODE, "run_id": RUN_ID,
                "error": str(e),
            }
            append_result(task, err_row)
            print(f"  [{i}/{len(rows)}] {record['id']} FAILED: {e}")
    aggregate_csv()
    write_progress()

aggregate_csv()
write_progress()
write_run_metadata()
print("Done. RUN_DIR =", RUN_DIR)
print("No generated code was executed.")


In [ ]:
# Unload model and free GPU memory
import gc
import torch

for _name in ("model", "tokenizer"):
    try:
        del globals()[_name]
    except KeyError:
        pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.synchronize()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
print("Model unloaded. GPU memory released.")


## 11. Export ZIP + SHA256 (Kaggle Output pane)

Creates ZIP containing the whole results/ tree.

In [ ]:
# Export ZIP for Kaggle Output pane. No cloud drive, no browser download.
import shutil
import os as _os2

assert RESULTS_ROOT.startswith(WORKDIR + _os2.sep), RESULTS_ROOT
assert ZIP_DIR == WORKDIR, ZIP_DIR

aggregate_csv()
write_run_metadata()
write_progress()

# Write the export manifest, then hash everything (manifest included).
zip_name = f"slm-benchmark_results_{MODE}_{RUN_ID}.zip"
_manifest = {
    "run_id": RUN_ID, "mode": MODE, "model_id": MODEL_ID, "short": SHORT,
    "model_dir": MODEL_DIR, "zip_name": zip_name, "platform": PLATFORM,
    "results_root": RESULTS_ROOT, "zip_dir": ZIP_DIR,
    "execution_backend": EXECUTION_BACKEND,
    "generated_code_executed": ALLOW_UNSANDBOXED_CODE_EXECUTION,
    "created_utc": datetime.now(timezone.utc).isoformat(),
}
with open(_os2.path.join(METADATA_DIR, "export_manifest.json"), "w", encoding="utf-8") as fh:
    import json as _j2
    _j2.dump(_manifest, fh, ensure_ascii=False, indent=2)
    fh.write("\n")

sums_path, entries = write_sha256sums(RESULTS_ROOT)
print("Wrote", sums_path, f"({len(entries)} files)")
shutil.copyfile(sums_path, _os2.path.join(METADATA_DIR, "SHA256SUMS.txt"))

print("\nResult files under RESULTS_ROOT =", RESULTS_ROOT)
for _dirpath, _dirs, _fnames in _os2.walk(RESULTS_ROOT):
    for _fn in sorted(_fnames):
        _p = _os2.path.join(_dirpath, _fn)
        print(" ", _os2.path.relpath(_p, RESULTS_ROOT), _os2.path.getsize(_p), "bytes")

# Place the ZIP next to RESULTS_ROOT (in /kaggle/working/) so it is not inside
# the tree being archived. It contains a top-level results/ directory.
zip_path = _os2.path.join(ZIP_DIR, zip_name)
_results_parent = _os2.path.dirname(RESULTS_ROOT.rstrip("/")) or "."
_results_base = _os2.path.basename(RESULTS_ROOT.rstrip("/"))
shutil.make_archive(zip_path[:-4], "zip", root_dir=_results_parent, base_dir=_results_base)
print("Zipped:", zip_path, _os2.path.getsize(zip_path), "bytes")

zip_hash = sha256_file(zip_path)
print("ZIP SHA256 =", zip_hash)
sidecar = zip_path + ".sha256"
with open(sidecar, "w", encoding="utf-8", newline="\n") as fh:
    fh.write(zip_hash + "  " + zip_name + "\n")
print("Sidecar:", sidecar)

print("Kaggle: open the Output pane on the right and download:", zip_name)
print("Also download the .sha256 sidecar for verification.")
print("No browser download is triggered from the notebook.")
